# Lab 03 : Anatomie Parquet, Nettoyage Silver Partie 1 & Typage Strict
**Cours :** Business Intelligence and Big Data Analysis (`01 31 0367 00`)  
**Date & Format :** Lab #3 (EN : Lun 23.11.2026 après-midi *après le Tutorial #2* / FR : Mar 24.11.2026) · 3 Heures (180 min)

---

## 🎯 1. Contexte & Architecture-as-Code
Lors du Lab 02, vous avez figé les données brutes telles quelles dans `bronze_staged/*.parquet`. Avant d'appliquer les filtres statistiques d'anomalies (Lab 04), cette séance relie directement **l'anatomie interne d'Apache Parquet vue au Cours 02** à **l'hygiène structurelle Silver et au typage strict des schémas**, en cohérence avec vos **règles DAMA (`DQR`)** du Tutorial #2 :

```mermaid
flowchart TD
    B0["2. Couche Bronze : bronze_staged/*.parquet"] --> P0["3. Anatomie Parquet & Pushdown (3.1 - 3.4)"]
    P0 --> Q0["4. Profilage Qualité Bronze & Scorecard (4.1 - 4.3)"]
    Q0 --> S1["5. Hygiène des chaînes & table d'alias canoniques"]
    S1 --> S2["6. Typage strict, parsing dates, déduplication & bascule S3 (6.1 - 6.2)"]
    S2 --> P1["7. Contrats de schéma Silver -> silver_part1/*.parquet"]
```

**Vos objectifs :**
1. **Explorer les entrailles d'Apache Parquet (Cours 02) :** Vérifier les octets magiques `PAR1` en en-tête et pied de page, décoder la taille du footer, inspecter les *Row Groups*, *Column Chunks*, décalages d'octets, l'encodage `RLE_DICTIONARY`/`RLE`/`SNAPPY`, les statistiques Min/Max/Null du footer et les mécanismes **Projection & Predicate Pushdown**.
2. **Profiler le Bronze (lecture seule) :** Construire votre **Scorecard DAMA DQ Bronze**.
3. **Hygiène des chaînes & alias :** Corriger les espaces parasites, la casse mixte (`'Press_01 '`), les alias (`'P-02'` -> `'PRESS_02'`) et `downtime_type` (`'unplanned'` -> `'Unplanned'`).
4. **Comprendre & appliquer les types Silver :** Démontrer pourquoi laisser des horodatages ou dates en chaînes casse l'arithmétique, le tri, la RAM et les filtres SQL ; typer les colonnes temporelles (`timestamp[ns]`, `timestamp[ns, tz=UTC]`, `date32[day]`), dédupliquer (`keep='last'`) et corriger le **basculement de minuit `S3` (`22:00–06:00`)** via `(timestamp - 6h)`.
5. **Persister & auditer les contrats de schéma :** Sauvegarder les 5 tables typées dans `silver_part1/*.parquet` et valider leurs schémas Arrow/Parquet ainsi que DuckDB `DESCRIBE`.

---

## ⚙️ 2. Configuration de l'environnement & découverte des fichiers Bronze

### Étape 1 : Localiser les entrées Bronze, PyArrow/DuckDB & créer `silver_part1`

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = None
for _cand in [_cwd, *_cwd.parents]:
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break
assert REPO_ROOT is not None, f"Could not locate 03_labs_student_starters from {_cwd}"

CASE_NAME = "case1_smartpress_automotive"
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME


def resolve_stage_dir(stage_name: str) -> Path:
    """Read upstream stage from student sandbox_outputs if populated, else fallback read-only to reference solution."""
    cand = SANDBOX_DATA / stage_name
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Stage '{stage_name}' not found in {cand} (and no reference fallback exists). Run the prerequisite Lab notebook first."
    )
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

BRONZE_STAGED = resolve_stage_dir("bronze_staged")
SILVER_P1 = SANDBOX_DATA / "silver_part1"
SILVER_P1.mkdir(parents=True, exist_ok=True)

df_telem = pd.read_parquet(BRONZE_STAGED / "bronze_telemetry.parquet")
df_cmms = pd.read_parquet(BRONZE_STAGED / "bronze_maintenance.parquet")
df_machines = pd.read_parquet(BRONZE_STAGED / "bronze_machines.parquet")
df_shifts = pd.read_parquet(BRONZE_STAGED / "bronze_shifts.parquet")
df_variants = pd.read_parquet(BRONZE_STAGED / "bronze_product_variants.parquet")

print("Bronze source :", BRONZE_STAGED)
print("Silver Part 1 :", SILVER_P1)
print(f"Loaded telemetry: {len(df_telem):,} rows | CMMS tickets: {len(df_cmms):,} rows")


## 📦 3. Anatomie d'Apache Parquet — En-tête, Pied de page & Pushdown (Cours 02)
Au **Cours 02 (§4.4)**, vous avez étudié pourquoi Apache Parquet surpasse CSV/JSON en analytique. Ouvrons `bronze_telemetry.parquet` comme fichier binaire brut puis avec `pyarrow.parquet.ParquetFile` — **sans charger les lignes** — pour inspecter chaque composant de sa structure interne :

```text
 [0 .. 4)              En-tête magique : b'PAR1' (4 octets)
 [4 .. debut_footer)   Row Groups -> Column Chunks -> Pages (Dictionnaire + RLE + Snappy)
 [debut_footer .. -8)  Pied de page / Footer (Thrift : schéma, offsets, stats min/max/nulls)
 [-8 .. -4)            Entier 4 octets (little-endian) = Taille du Footer en octets
 [-4 .. fin)           Pied de page magique : b'PAR1' (4 octets)
```

### 3.1 Démonstration : Octets magiques (`PAR1`), taille du Footer, métadonnées & schéma embarqué
**Pourquoi le Footer est-il écrit à la fin mais lu EN PREMIER ?** Parce qu'à l'écriture, les décalages d'octets et bornes min/max ne sont connus qu'une fois la dernière ligne traitée. À la lecture, le moteur saute directement à `EOF - 8`, lit la taille sur 4 octets, puis charge le Footer avant de décompresser la moindre page de données !

*Exécutez directement la cellule de démonstration ci-dessous pour inspecter les octets bruts et le schéma Arrow embarqué.*

In [ ]:
# Démonstration prête à exécuter : inspecter la structure binaire et le footer d'un fichier Parquet
pq_path = BRONZE_STAGED / "bronze_telemetry.parquet"

# Lire le fichier en octets bruts pour vérifier l'en-tête et le pied de page binaires
raw_bytes = pq_path.read_bytes()

# 1. Vérifier la signature magique "PAR1" sur les 4 premiers et les 4 derniers octets
magic_header = raw_bytes[:4]
magic_footer = raw_bytes[-4:]

# 2. Les 4 octets juste avant le "PAR1" final (indices -8..-4) contiennent la taille du Footer
#    encodée en entier little-endian (octet de poids faible en premier)
footer_len = int.from_bytes(raw_bytes[-8:-4], "little")

# 3. Ouvrir UNIQUEMENT les métadonnées (lit le Footer sans décompresser les pages de données !)
pf = pq.ParquetFile(pq_path)

# Contrôles : signatures PAR1 valides et taille du footer identique à celle lue par PyArrow
assert magic_header == b"PAR1" and magic_footer == b"PAR1", "Signature magique PAR1 invalide !"
assert footer_len == pf.metadata.serialized_size, "Taille de footer incohérente !"

print(f"Taille fichier    : {len(raw_bytes):,} octets")
print(f"En-tête / Fin     : {magic_header} ... {magic_footer}")
print(f"Taille du Footer  : {footer_len:,} octets (identique à pf.metadata.serialized_size)")
print(f"Lignes/Cols/RGs   : {pf.metadata.num_rows:,} lignes | {pf.metadata.num_columns} cols | {pf.metadata.num_row_groups} Row Group(s)")
print(f"Format / Outil    : v{pf.metadata.format_version} | {pf.metadata.created_by}")
print(f"Clés métadonnées  : {list((pf.metadata.metadata or {}).keys())}")
print("\nSchéma Arrow embarqué (notez que 'timestamp' & '_ingestion_timestamp_utc' sont encore 'string' en Bronze !) :")
print(pf.schema_arrow)


### 3.2 Démonstration : Column Chunks, offsets mémoire, encodages (`Dictionnaire` + `RLE` + `Snappy`) & statistiques du Footer
Exécutez cette démonstration pour inspecter `pf.metadata.row_group(0)` colonne par colonne **uniquement depuis les métadonnées du Footer** (sans lire les pages de données) et observer :
1. **Disposition binaire en colonnes :** Chaque *Column Chunk* suit exactement le précédent (`dictionary_page_offset` / `data_page_offset`), et `4 + sum(comp_B) + footer_len + 8` égale la **taille exacte du fichier à l'octet près** !
2. **Compression Dictionnaire + RLE :** `_source_file` et `_ingestion_timestamp_utc` (`23 164` chaînes identiques) pèsent seulement ~`200–265 octets` !
3. **Audit Qualité gratuit depuis le Footer :** Les statistiques `nulls`, `min` et `max` révèlent déjà `120` valeurs nulles, `-999.0`/`4850.0 Nm`, `-50` unités, `'P-02'`..`'press_03'` et `'2099-01-01'` *sans décompresser une seule ligne* !

In [ ]:
# Démonstration prête à exécuter : lire l'agencement et les stats min/max/nulls depuis le Footer
rg = pf.metadata.row_group(0)

chunk_rows = []
for i in range(rg.num_columns):
    # c = descripteur de la colonne i ; st = statistiques pré-calculées (min, max, null_count)
    c = rg.column(i)
    st = c.statistics
    chunk_rows.append({
        "column": c.path_in_schema,                 # Nom de colonne dans le schéma
        "phys_type": c.physical_type,               # Type physique Parquet (INT64, DOUBLE, BYTE_ARRAY)
        "comp": c.compression,                      # Algorithme de compression (SNAPPY)
        "encodings": ",".join(c.encodings),         # Encodages (Dictionnaire + RLE + PLAIN)
        "dict_off": c.dictionary_page_offset,       # Offset en octets de la page dictionnaire
        "data_off": c.data_page_offset,             # Offset en octets des pages de données
        "raw_B": c.total_uncompressed_size,         # Taille non compressée (octets)
        "comp_B": c.total_compressed_size,          # Taille compressée sur disque (octets)
        "nulls": st.null_count if st else None,     # Nombre de valeurs manquantes (lu dans le footer)
        "min_stat": repr(st.min) if st and st.has_min_max else None,  # Valeur minimale du Row Group
        "max_stat": repr(st.max) if st and st.has_min_max else None,  # Valeur maximale du Row Group
    })

df_chunks = pd.DataFrame(chunk_rows)

# Vérifier l'équation exacte du fichier :
# 4o ("PAR1") + somme des Column Chunks compressés + Footer Thrift + 8o finaux (4o taille + 4o "PAR1")
total_payload_b = int(df_chunks["comp_B"].sum())
assert 4 + total_payload_b + footer_len + 8 == len(raw_bytes), "Écart sur la carte binaire !"

print(f"Carte binaire exacte : 4o (PAR1) + {total_payload_b:,}o (Chunks) + {footer_len:,}o (Footer) + 8o (len+PAR1) = {len(raw_bytes):,}o")
print("\nMétadonnées des Column Chunks & Statistiques du Footer (Row Group 0) :")
print(df_chunks.to_string(index=False))


### 3.3 Démonstration : Projection Pushdown (`columns=[...]`)
Ne lire que `["machine_id", "torque_Nm"]` utilise les offsets du Footer pour sauter directement à ces 2 colonnes et ignorer >80 % des octets Parquet (et >96 % par rapport au CSV brut !).

In [ ]:
# Démonstration prête à exécuter : Projection Pushdown (ignorer les colonnes inutiles via le Footer)
selected_cols = ["machine_id", "torque_Nm"]
csv_bytes = (BRONZE_RAW / "sensor_telemetry.csv").stat().st_size if (BRONZE_RAW / "sensor_telemetry.csv").exists() else 0

# PyArrow ne lit QUE les 2 colonnes demandées sur les 12 du fichier
tbl_proj = pq.read_table(pq_path, columns=selected_cols)

# Additionner uniquement la taille compressée des 2 Column Chunks ciblés (depuis le Footer)
proj_bytes = sum(
    rg.column(i).total_compressed_size
    for i in range(rg.num_columns)
    if rg.column(i).path_in_schema in selected_cols
)
print(f"Projection Pushdown ({selected_cols}) :")
print(f"  CSV brut : {csv_bytes:,} o -> Tous chunks Parquet : {total_payload_b:,} o -> 2 cols projetées : {proj_bytes:,} o ({100 * (1 - proj_bytes / total_payload_b):.1f} % I/O évités !)")
print(f"  Dimensions table projetée : {tbl_proj.num_rows:,} lignes x {tbl_proj.num_columns} cols")


### 3.4 Démonstration : Predicate Pushdown multi-Row-Groups (`filters=[...]`)
Sur un fichier découpé en plusieurs *Row Groups* (`row_group_size=6000`), le moteur examine les bornes `[min, max]` de chaque footer et **saute des Row Groups entiers** incompatibles ! Remarquez aussi que `'press_03'` (minuscule) se retrouve isolé dans le Row Group 3 (`'p' > 'P'` en ASCII), montrant pourquoi l'hygiène de la Section 5 est indispensable au filtrage.

In [ ]:
# Démonstration prête à exécuter : Predicate Pushdown (saut de Row Groups via les stats [min, max])
# Écrire un buffer Parquet en mémoire trié par machine_id et découpé en 4 Row Groups
sink = pa.BufferOutputStream()
pq.write_table(
    pa.Table.from_pandas(df_telem.sort_values("machine_id"), preserve_index=False),
    sink,
    row_group_size=6000,   # Découpe les 23 164 lignes en 4 Row Groups (~6 000 lignes chacun)
    compression="snappy",
)
multi_buf = sink.getvalue()
pf_multi = pq.ParquetFile(pa.BufferReader(multi_buf))

# Lecture filtrée : PyArrow vérifie [min, max] de chaque Row Group dans le footer AVANT de décompresser
tbl_pred = pq.read_table(
    pa.BufferReader(multi_buf),
    columns=["timestamp", "machine_id", "torque_Nm"],
    filters=[("machine_id", "=", "PRESS_03")],
)

print(f"Statistiques multi-Row-Groups ({pf_multi.metadata.num_row_groups} Row Groups triés par machine_id) :")
for r in range(pf_multi.metadata.num_row_groups):
    rg_r = pf_multi.metadata.row_group(r)
    st_m = rg_r.column(1).statistics  # colonne 1 = machine_id
    # Si "PRESS_03" est hors de [min, max], le Row Group entier est ignoré sans décompression
    can_skip = not (st_m.min <= "PRESS_03" <= st_m.max)
    print(f"  RG {r} : lignes={rg_r.num_rows:>4} | machine_id=[{st_m.min!r}, {st_m.max!r}] -> {'IGNORÉ via Footer !' if can_skip else 'CANDIDAT'}")
print(f"Résultat Predicate Pushdown ('PRESS_03') : {tbl_pred.num_rows:,} lignes ('press_03' non nettoyé en RG 3 a été manqué !)")


## 🔍 4. Profilage Qualité Bronze Initial (Strictement en Lecture Seule !)
**Règle d'Or (`Immutabilité du Bronze`) :** Le footer Parquet nous a déjà alertés sur les identifiants sales, `120` valeurs nulles, couples extrêmes et dates `2099-*`. Auditez maintenant les tables Bronze (`df_telem`, `df_cmms`, `df_machines`) pour quantifier 100 % des anomalies selon nos **règles DAMA DQ**.

### 4.1 Scan DQ : Dérives d'Identifiants & de Casse (`machine_id` & `downtime_type`)
Confrontez les identifiants bruts au référentiel ERP officiel (`PRESS_01..PRESS_04`).

In [ ]:
valid_machines = set(df_machines["machine_id"].astype(str).str.strip().str.upper())

# À FAIRE : Inspectez `machine_id` brut avec repr() (espaces parasites) et `downtime_type` GMAO
raw_machine_counts = None     # REMPLACER PAR : df_telem["machine_id"].value_counts(dropna=False)
cmms_type_counts = None       # REMPLACER PAR : df_cmms["downtime_type"].value_counts(dropna=False)

if raw_machine_counts is not None and cmms_type_counts is not None:
    print("Machines officielles ERP :", sorted(valid_machines))
    print("\n1. Valeurs brutes machine_id SCADA ('Press_01 ', 'P-02', 'press_03' & orphelin 'PRESS_99') :")
    for k, cnt in raw_machine_counts.items():
        print(f"  {repr(k):<14} -> {cnt:>6,} lignes")
    print("\n2. Dérives de casse CMMS downtime_type ('Unplanned' vs 'unplanned' vs 'Preventive') :")
    print(cmms_type_counts.to_string())
else:
    print("⏳ [À FAIRE] Assignez raw_machine_counts et cmms_type_counts ci-dessus.")


### 4.2 Scan DQ : Bornes Physiques, Nuls (`NaN`), Dates Futures & Doublons
Mobilisez les **masques booléens vectorisés** (`<`, `>`, `|`, `~`, `.isin()`) pour quantifier :
1. Couples mécaniques impossibles (`< 0` ex. `-999.0 Nm` ou `> 200.0` ex. `4850.0 Nm`).
2. Températures manquantes (`NaN` sur `process_temp_K`) — rappel : **ne jamais faire `.dropna()`** sous peine de perdre les unités produites !
3. `scrap_units > produced_units`, horodatages `2099-*`, doublons MQTT et tickets CMMS orphelins (`CMMS-20269999`).

In [ ]:
# À FAIRE : Complétez les compteurs ci-dessous via des masques booléens vectoriels Pandas
bad_torque_count = None   # REMPLACER PAR : ((df_telem["torque_Nm"] < 0) | (df_telem["torque_Nm"] > 200)).sum()
null_temp_count = None    # REMPLACER PAR : df_telem["process_temp_K"].isna().sum()
dupes_count = None        # REMPLACER PAR : df_telem.duplicated(subset=["timestamp", "machine_id", "product_variant"]).sum()

tel_ts = pd.to_datetime(df_telem["timestamp"], errors="coerce")
future_tel_count = (tel_ts > "2027-01-01").sum()
bad_scrap_count = (df_telem["scrap_units"] > df_telem["produced_units"]).sum()

mnt_ts = pd.to_datetime(df_cmms["event_date"], errors="coerce")
mnt_bad_mask = (~df_cmms["machine_code"].astype(str).str.strip().str.upper().isin(valid_machines)) | mnt_ts.isna() | (mnt_ts > "2027-01-01")

if all(v is not None for v in (bad_torque_count, null_temp_count, dupes_count)):
    print(f"Télémesure couple hors bornes (<0 ou >200 Nm) : {bad_torque_count}")
    print(f"Télémesure process_temp_K manquant (NaN)     : {null_temp_count}  (interpolation au Labo 04, PAS de dropna !)")
    print(f"Télémesure doublons MQTT                     : {dupes_count}")
    print(f"Télémesure dates futures 2099-*              : {future_tel_count}")
    print(f"Télémesure scrap_units > produced_units      : {bad_scrap_count}")
    print("\nTickets CMMS signalés (alias non canoniques, presse orpheline ou date 2099-*) :")
    display(df_cmms.loc[mnt_bad_mask, ["ticket_id", "event_date", "machine_code", "downtime_type", "failure_mode", "repair_duration_min"]].head(8))
    print("  Total tickets signalés :", mnt_bad_mask.sum(), "| Contrôle défensif repair_duration_min <= 0 :", (df_cmms["repair_duration_min"] <= 0).sum())
else:
    print("⏳ [À FAIRE] Complétez bad_torque_count, null_temp_count et dupes_count ci-dessus.")


### 4.3 Scorecard DAMA DQ Bronze (Plan d'Action Labos 03–04)
Synthétisez vos constats en un **`DataFrame` Scorecard** reliant chaque anomalie à son traitement **Labo 03 (Normaliser/Dédupliquer)** ou **Labo 04 (Interpoler/Quarantaine)**.

In [ ]:
if all(v is not None for v in (bad_torque_count, null_temp_count, dupes_count)):
    orphan_tel_count = (df_telem["machine_id"].astype(str).str.strip().str.upper() == "PRESS_99").sum()
    alias_tel_count = df_telem["machine_id"].isin(["Press_01 ", "P-02", "press_03"]).sum()
    cmms_future_orphan_count = (mnt_ts > "2027-01-01").sum()
    cmms_unplanned_lower_count = (df_cmms["downtime_type"] == "unplanned").sum()

    dq_scorecard = pd.DataFrame([
        {"Rule_ID": "DQR-KEY", "Source": "SCADA + CMMS", "Anomalie": "Dérives espaces/casse/alias ('Press_01 ', 'P-02', 'unplanned')", "Nombre": alias_tel_count + cmms_unplanned_lower_count, "Action_Planifiee": "Labo 03 Normaliser (.strip().upper() + alias)"},
        {"Rule_ID": "DQR-107", "Source": "SCADA CSV", "Anomalie": "Doublons MQTT télémétrie", "Nombre": dupes_count, "Action_Planifiee": "Labo 03 Dédupliquer (keep='last')"},
        {"Rule_ID": "DQR-101", "Source": "SCADA + CMMS", "Anomalie": "Actif fantôme orphelin (PRESS_99)", "Nombre": orphan_tel_count, "Action_Planifiee": "Labo 04 Quarantaine (BLOQUANT)"},
        {"Rule_ID": "DQR-102", "Source": "SCADA + CMMS", "Anomalie": "Dates futures 2099-* (incl. CMMS-20269999)", "Nombre": future_tel_count + cmms_future_orphan_count, "Action_Planifiee": "Labo 04 Quarantaine"},
        {"Rule_ID": "DQR-103", "Source": "SCADA CSV", "Anomalie": "Couple impossible torque_Nm (-999 / 4850 Nm)", "Nombre": bad_torque_count, "Action_Planifiee": "Labo 04 Quarantaine"},
        {"Rule_ID": "DQR-104", "Source": "SCADA CSV", "Anomalie": "scrap_units > produced_units", "Nombre": bad_scrap_count, "Action_Planifiee": "Labo 04 Quarantaine"},
        {"Rule_ID": "DQR-TEMP", "Source": "SCADA CSV", "Anomalie": "process_temp_K manquant (NaN)", "Nombre": null_temp_count, "Action_Planifiee": "Labo 04 Interpolation temporelle (PAS de dropna!)"},
    ])
    print("=== SCORECARD QUALITÉ BRONZE (PLAN D'ACTION) ===")
    display(dq_scorecard)
else:
    print("⏳ Complétez les scans DQ ci-dessus pour générer la Scorecard DQ Bronze.")


## 🧹 5. Hygiène des chaînes & table d'alias canoniques
Normalisez `machine_id`, `machine_code` et `downtime_type` de façon déterministe avant toute déduplication.

In [ ]:
# Table de correspondance des alias hérités -> identifiants ERP canoniques
MACHINE_ALIASES = {
    "P-01": "PRESS_01", "PRESS-01": "PRESS_01", "P_01": "PRESS_01",
    "P-02": "PRESS_02", "PRESS-02": "PRESS_02", "P_02": "PRESS_02",
    "P-03": "PRESS_03", "PRESS-03": "PRESS_03", "P_03": "PRESS_03",
    "P-04": "PRESS_04", "PRESS-04": "PRESS_04", "P_04": "PRESS_04",
}

def clean_machine_code(val):
    if pd.isna(val):                              # Protéger contre les valeurs manquantes / NaN
        return "UNKNOWN"
    s = str(val).strip().upper()                  # Supprimer les espaces ('Press_01 ') et passer en majuscules ('press_03')
    # dict.get(cle, defaut) prend 2 paramètres :
    #   1er arg (s) : la clé à chercher dans MACHINE_ALIASES (ex. 'P-02' -> 'PRESS_02')
    #   2e arg  (s) : la valeur de repli retournée si la clé n'est PAS dans MACHINE_ALIASES (garde 'PRESS_01' ou 'PRESS_99' tel quel)
    return MACHINE_ALIASES.get(s, s)

def standardize_downtime_type(val):
    s = str(val).strip().lower()                  # Supprimer les espaces et passer en minuscules
    if "prevent" in s or "pm" in s:               # Couvrir 'Preventive', 'preventive', 'PM', etc.
        return "Preventive"
    if "correct" in s:                            # Couvrir 'Corrective', 'corrective_action', etc.
        return "Corrective"
    return "Unplanned"                            # Catégorie par défaut ('unplanned', 'Unplanned', etc.)

# Vérifier les valeurs brutes AVANT nettoyage
print("Identifiants bruts :", sorted(df_telem["machine_id"].dropna().unique()))
print("Types d'arrêt bruts :", df_cmms["downtime_type"].value_counts().to_dict())

# À FAIRE : appliquer clean_machine_code et standardize_downtime_type à df_telem, df_cmms, df_machines
#   - df_machines["machine_id"]  <- clean_machine_code
#   - df_telem["machine_id"]     <- clean_machine_code
#   - df_cmms["machine_code"]    <- clean_machine_code
#   - df_cmms["downtime_type"]   <- standardize_downtime_type


# Vérifier les valeurs nettoyées APRÈS vos appels à .apply(...)
print("Identifiants propres :", sorted(df_telem["machine_id"].unique()))
print("Types d'arrêt propres :", df_cmms["downtime_type"].value_counts().to_dict())


## ⏱️ 6. Typage strict, horodatages, déduplication & bascule de nuit `S3`
### 6.1 Pourquoi le typage strict est vital en Parquet Silver (`string` vs `timestamp` / `date`)
Lors du Lab 02, `bronze_telemetry.parquet` a volontairement conservé `timestamp` et `_ingestion_timestamp_utc` en **chaînes brutes (`string` / `object`)**. Avant de les convertir, réalisez trois expériences concrètes démontrant **pourquoi conserver des dates ou nombres en texte en Silver casse les pipelines** :
1. **Arithmétique & accesseurs impossibles :** `timestamp - pd.Timedelta(hours=6)` lève une `TypeError`, `.dt.hour` échoue (`AttributeError`), et `.interpolate(method='time')` (Lab 04) ou `.resample('1h')` (Lab 05) sont impossibles.
2. **Pièges silencieux du tri alphabétique :** Les chaînes comparent les codes ASCII caractère par caractère, pas la chronologie (`'09/10/2026' > '01/11/2026'` vaut `True` ; `'2026-10-01T06:00:00' > '2026-10-01 07:00:00'` vaut `True` car `'T' > ' '` !) — faussant les filtres SQL et l'élagage Parquet `min/max` !
3. **Mémoire RAM & contrat moteur :** `datetime64[ns]` occupe `8 octets/ligne` au lieu d'objets texte Python (**~8,5× moins de RAM**), et expose directement des types `TIMESTAMP` / `DATE` natifs à DuckDB (`INTERVAL '2 HOURS'` au Lab 06) et aux outils BI !

In [ ]:
# Expérience 1 : L'arithmétique temporelle échoue sur une colonne texte
try:
    _ = df_telem["timestamp"].head(3) - pd.Timedelta(hours=6)
except TypeError as err:
    print(f"[Expérience 1] Calcul sur texte ÉCHOUÉ comme prévu -> TypeError : {err}")

# Expérience 2 : Pièges silencieux du tri alphabétique vs chronologique
eu_cmp_str = "09/10/2026" > "01/11/2026"
iso_cmp_str = "2026-10-01T06:00:00" > "2026-10-01 07:00:00"  # 'T' (ASCII 84) > ' ' (ASCII 32) !
num_cmp_str = "9" > "10"
print(f"[Expérience 2] Pièges texte -> '09/10/2026' > '01/11/2026' = {eu_cmp_str}, '2026-10-01T06:00:00' > '2026-10-01 07:00:00' = {iso_cmp_str}, '9' > '10' = {num_cmp_str} !")

# À FAIRE : Mesurer la RAM utilisée par df_telem['timestamp'] en texte (object) vs datetime64[ns]
ram_str_bytes = None  # REMPLACER PAR : df_telem["timestamp"].memory_usage(deep=True)
ram_dt_bytes = None   # REMPLACER PAR : pd.to_datetime(df_telem["timestamp"], errors="coerce").memory_usage(deep=True)

if ram_str_bytes is not None and ram_dt_bytes is not None:
    print(f"[Expérience 3] RAM pour 'timestamp' -> object/str : {ram_str_bytes:,} o vs datetime64[ns] : {ram_dt_bytes:,} o ({ram_str_bytes / ram_dt_bytes:.2f}x plus léger !)")
else:
    print("⏳ [À FAIRE] Calculez ram_str_bytes et ram_dt_bytes ci-dessus.")


### 6.2 Parsing des dates, déduplication post-normalisation & bascule de l'équipe de nuit `S3`
1. Convertissez `df_telem['timestamp']` et `df_cmms['event_date']` en **`datetime64[ns]`** natif (`pd.to_datetime(..., errors='coerce')`).
2. Convertissez également `_ingestion_timestamp_utc` en **`datetime64[ns, UTC]`** sur les 5 tables (comme annoncé au Lab 02 !).
3. Dédupliquez la télémétrie et la GMAO **après** normalisation (`keep='last'`).
4. Calculez `shift_code` et **`operational_shift_date = (ts - 6h).dt.date`** — conservé en vrai type **`datetime.date`** (Parquet `DATE` / `date32[day]`, **sans** `.astype(str)` !) pour rattacher `00:00–05:45` à la bonne journée opérationnelle.

In [ ]:
# À FAIRE :
# 1. Convertir df_telem['timestamp'] et df_cmms['event_date'] via pd.to_datetime(..., errors='coerce')
# 2. Dédupliquer df_telem (sur ["timestamp", "machine_id", "product_variant"]) et df_cmms (sur ["ticket_id"]) avec keep='last'
# 3. Convertir _ingestion_timestamp_utc en datetime64[ns, UTC] sur les 5 DataFrames (format="ISO8601", utc=True, errors="coerce")
# 4. Calculer shift_code ('S1' 06-14h, 'S2' 14-22h, 'S3' 22-06h) et operational_shift_date = (ts - pd.Timedelta(hours=6)).dt.date (SANS .astype(str) !)

if pd.api.types.is_datetime64_any_dtype(df_telem["timestamp"]):
    print("Type de timestamp télémétrie :", df_telem["timestamp"].dtype)
else:
    print("⏳ [À FAIRE] Parsez les dates, dédupliquez et calculez shift_code + operational_shift_date ci-dessus.")


## 💾 7. Contrats de schéma Silver, sauvegarde `silver_part1/*.parquet` & audit des types
Avant d'écrire dans Silver Partie 1, appliquez un **Contrat de Schéma Silver** explicite sur les 5 tables :
- Colonnes textuelles (identifiants/libellés) -> **`string`** (fini les colonnes `object` ambiguës !)
- Mesures & compteurs numériques -> **`float64`** / **`int64`** explicites
- Horodatages opérationnels (`timestamp`, `event_date`) -> **`timestamp[ns]`** (`INT64` dans Parquet)
- Lignage d'ingestion (`_ingestion_timestamp_utc`) -> **`timestamp[ns, tz=UTC]`**
- Date opérationnelle (`operational_shift_date`) -> **`date32[day]`** (`DATE`, `INT32` dans Parquet)

Après écriture, comparez les **schémas Parquet Bronze vs Silver Partie 1**, exécutez `DESCRIBE` sous DuckDB et vérifiez que le footer Parquet stocke désormais de vraies bornes **`Timestamp`** et **`Date`** !

In [ ]:
# Contrat de schéma Silver explicite pour les 5 tables (dates/timestamps déjà typés à la section 6.2)
SILVER_SCHEMAS = {
    "telemetry": {
        "string": ["machine_id", "product_variant", "shift_code", "_source_file"],
        "float64": ["air_temp_K", "process_temp_K", "torque_Nm"],
        "int64": ["rotational_speed_rpm", "tool_wear_min", "produced_units", "scrap_units"],
    },
    "maintenance": {
        "string": ["ticket_id", "machine_code", "downtime_type", "failure_mode", "shift_code", "_source_file"],
        "int64": ["repair_duration_min"],
    },
    "machines": {
        "string": ["machine_id", "machine_name", "drive_type", "_source_file"],
        "float64": ["nominal_cycle_time_sec", "hourly_operating_cost_eur"],
    },
    "shifts": {
        "string": ["shift_id", "shift_name", "start_time", "end_time", "_source_file"],
        "int64": ["planned_duration_hours"],
    },
    "variants": {
        "string": ["variant_code", "variant_name", "material_grade", "_source_file"],
        "float64": ["unit_scrap_cost_eur"],
        "int64": ["target_hourly_output"],
    },
}

def enforce_dtypes(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    out = df.copy()  # Travailler sur une copie indépendante sans modifier le DataFrame source
    for col in spec.get("string", []):
        out[col] = out[col].astype("string")                                  # Type StringDtype Pandas (fini 'object' ambigu)
    for col in spec.get("float64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64") # Texte invalide -> NaN, puis flottant 64 bits
    for col in spec.get("int64", []):
        out[col] = pd.to_numeric(out[col], errors="coerce").astype("int64")   # Entier 64 bits
    return out

# 1. Appliquer le contrat de schéma Silver aux 5 DataFrames
df_telem = enforce_dtypes(df_telem, SILVER_SCHEMAS["telemetry"])
df_cmms = enforce_dtypes(df_cmms, SILVER_SCHEMAS["maintenance"])
df_machines = enforce_dtypes(df_machines, SILVER_SCHEMAS["machines"])
df_shifts = enforce_dtypes(df_shifts, SILVER_SCHEMAS["shifts"])
df_variants = enforce_dtypes(df_variants, SILVER_SCHEMAS["variants"])

# 2. Sauvegarder les 5 tables typées dans silver_part1/*.parquet (index=False évite la colonne __index_level_0__)
outputs = {
    "silver_part1_telemetry.parquet": df_telem,
    "silver_part1_maintenance.parquet": df_cmms,
    "silver_machines.parquet": df_machines,
    "silver_shifts.parquet": df_shifts,
    "silver_product_variants.parquet": df_variants,
}
for fname, df_out in outputs.items():
    df_out.to_parquet(SILVER_P1 / fname, index=False, compression="snappy")

# 3. Vérifier l'hygiène des chaînes avant l'audit des schémas
assert df_telem["machine_id"].str.contains(" ").sum() == 0, "Espaces restants dans machine_id !"
assert set(df_cmms["downtime_type"]).issubset({"Unplanned", "Preventive", "Corrective"}), "downtime_type non normalisé !"

# 4. Comparer les schémas Arrow Bronze vs Silver Partie 1 & les stats min/max du footer Parquet :
#    - Schéma Arrow (schema_arrow) : types logiques de haut niveau utilisés en mémoire par Pandas/PyArrow/DuckDB
#      (ex. string, timestamp[ns], timestamp[ns, tz=UTC], date32[day]), conservés dans les métadonnées.
#    - Type physique Parquet (physical_type) : primitive binaire bas niveau stockée sur disque dans les Column Chunks
#      (ex. timestamp[ns] -> INT64 sur disque, date32[day] -> INT32 sur disque, string -> BYTE_ARRAY).
pf_bronze = pq.ParquetFile(BRONZE_STAGED / "bronze_telemetry.parquet")
pf_silver = pq.ParquetFile(SILVER_P1 / "silver_part1_telemetry.parquet")
bronze_types = {f.name: str(f.type) for f in pf_bronze.schema_arrow}
silver_types = {f.name: str(f.type) for f in pf_silver.schema_arrow}
rg_s = pf_silver.metadata.row_group(0)
silver_stats = {rg_s.column(i).path_in_schema: rg_s.column(i) for i in range(rg_s.num_columns)}

comparison = pd.DataFrame([
    {
        "column": col,
        "bronze_arrow_type": bronze_types.get(col, "<new>"),
        "silver_arrow_type": silver_types[col],
        "parquet_phys_type": silver_stats[col].physical_type,
        "footer_min": repr(silver_stats[col].statistics.min) if silver_stats[col].statistics else None,
        "footer_max": repr(silver_stats[col].statistics.max) if silver_stats[col].statistics else None,
    }
    for col in silver_types
])
print("=== COMPARATIF SCHÉMA PARQUET BRONZE vs SILVER PARTIE 1 & STATS FOOTER ===")
print(comparison.to_string(index=False))

# 5. Vérifier les contrats temporels Arrow/Parquet sur les 5 fichiers sauvegardés
for fname in outputs:
    sc = pq.ParquetFile(SILVER_P1 / fname).schema_arrow
    names = set(sc.names)
    assert "__index_level_0__" not in names, f"Fuite d'index dans {fname} !"
    ing_t = sc.field("_ingestion_timestamp_utc").type
    assert pa.types.is_timestamp(ing_t) and ing_t.tz == "UTC", f"{fname}: _ingestion_timestamp_utc doit être UTC, reçu {ing_t}"

assert pa.types.is_timestamp(pf_silver.schema_arrow.field("timestamp").type)
assert pa.types.is_date(pf_silver.schema_arrow.field("operational_shift_date").type)
cmms_sc = pq.ParquetFile(SILVER_P1 / "silver_part1_maintenance.parquet").schema_arrow
assert pa.types.is_timestamp(cmms_sc.field("event_date").type)
assert pa.types.is_date(cmms_sc.field("operational_shift_date").type)

# 6. Vérifier que DuckDB détecte directement les types natifs TIMESTAMP / TIMESTAMPTZ / DATE
con = duckdb.connect()
duck_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SILVER_P1 / 'silver_part1_telemetry.parquet'}')").df()
con.close()
print("\nDuckDB DESCRIBE sur silver_part1_telemetry.parquet (TIMESTAMP / TIMESTAMPTZ / DATE natifs !) :")
print(duck_schema[["column_name", "column_type"]].to_string(index=False))

print(f"\n5 fichiers Silver Partie 1 sauvegardés et vérifiés dans : {SILVER_P1}")
print("REMARQUE : PRESS_99, dates 2099 et couples -999/4850 sont conservés ici pour la Quarantaine du Lab 04 !")
